## 系统项目：沿路锥行驶与障碍物检测

本 notebook 介绍系统项目 1:10 比例小车的一项仿真练习。你将在 Dream Gym 中构建沿路锥行驶的策略，观察其表现，再加入障碍物和 LiDAR 测量，以便设计并测试停车响应。

练习遵循实际工程流程：定义道路和车辆模型，选择可用观测，编写策略，运行仿真，检查结果，再改进策略。车辆和传感器设置只是仿真的起点。请利用结果提出团队项目中的问题和测试，再到实体小车上验证所需行为。

每个系统项目团队都可以确定自己的重点。沿路锥行驶和停车的示例展示了自主系统的一个部分如何与感知、执行结合，并经过设计和测试。

**Notebook 目录**

- **准备与导入：** 安装指定版本的 Dream Gym，并选择短时或完整仿真。
- **1. 定义路锥道路：** 构建路锥规则排列、单侧排列及不规则排列的路线。
- **2. 定义车辆参数：** 配置 1:10 比例小车的简化模型及行驶方向锁存机制。
- **3. 定义路锥检测配置：** 选择仿真相机向策略提供的观测。
- **4. 创建环境：** 组合道路、车辆、观测和初始条件。
- **5. 沿路锥行驶策略：** 将检测到的路锥位置转换为转向和驱动指令。
- **6. 仿真沿路锥行驶策略：** 查看轨迹、时间序列和动画。
- **7. 在道路上添加障碍物：** 放置路边物体和车道内障碍物。
- **8. 添加基于 LiDAR 的障碍物观测：** 配置朝前的障碍物扫描。
- **9. 创建含障碍物的新环境：** 将现有路锥观测与 LiDAR 测量结合。
- **10. 在障碍物环境中仿真：** 观察原策略忽略新传感器时会发生什么。
- **11. 练习：添加物体检测与停车：** 扩展策略，测试制动响应，并研究哪些 LiDAR 光束重要。
- **12. 性能指标与不确定性扫描：** 通过重复运行和改变条件来评估策略。


## 准备与导入


在 Colab 中，此单元会安装课程指定版本的软件包；在本地运行时不会执行安装。

In [1]:
import importlib
import os
from pathlib import Path
import site
import subprocess
import sys
import sysconfig

COURSE_SOURCE_REF = "v0.4.0"
COURSE_REPOSITORY = "https://gitlab.unimelb.edu.au/dream/dream-gym.git"
COLAB_CHECKOUT = Path("/content/dream-gym-current")
IS_COLAB = "google.colab" in sys.modules
package_was_loaded = any(
    name == "dreamgym" or name.startswith("dreamgym.")
    for name in sys.modules
)
setup_diagnostic = None

if IS_COLAB:
    if package_was_loaded:
        setup_diagnostic = "restart the Colab runtime before continuing"
    else:
        try:
            if COLAB_CHECKOUT.exists():
                if not (COLAB_CHECKOUT / ".git").is_dir():
                    raise RuntimeError("unverifiable checkout")
                origin = subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "remote", "get-url", "origin"),
                    capture_output=True, text=True, check=True,
                ).stdout.strip()
                if origin != COURSE_REPOSITORY:
                    raise RuntimeError("unexpected checkout origin")
                status = subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "status", "--porcelain"),
                    capture_output=True, text=True, check=True,
                ).stdout.strip()
                if status:
                    raise RuntimeError("checkout has local changes")
                subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "fetch", "--depth", "1", "origin", COURSE_SOURCE_REF),
                    capture_output=True, text=True, check=True,
                )
                subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "checkout", "--detach", "FETCH_HEAD"),
                    capture_output=True, text=True, check=True,
                )
            else:
                subprocess.run(
                    ("git", "clone", "--branch", COURSE_SOURCE_REF, "--single-branch", COURSE_REPOSITORY, str(COLAB_CHECKOUT)),
                    capture_output=True, text=True, check=True,
                )
            subprocess.run(
                (sys.executable, "-m", "pip", "install", "-e", "."),
                cwd=COLAB_CHECKOUT, capture_output=True, text=True, check=True,
            )
            site.addsitedir(sysconfig.get_paths()["purelib"])
            os.chdir(COLAB_CHECKOUT)
            importlib.invalidate_caches()
        except (OSError, RuntimeError, subprocess.CalledProcessError):
            setup_diagnostic = "notebook setup failed"

if setup_diagnostic is not None:
    print(setup_diagnostic)


In [2]:
if setup_diagnostic is not None:
    raise RuntimeError(
        f"Notebook setup did not complete: {setup_diagnostic}"
    )


**运行模式**

`RUN_MODE` 只接受 `smoke` 或 `full`，默认值为 `smoke`。`OUTPUT_ROOT` 是相对于仓库根目录的路径，所有生成的文件都保存在该路径之下。

| 属性 | 单位 | 允许值 | 默认值或继承方式 | 用途 |
| --- | --- | --- | --- | --- |
| `RUN_MODE` | 无 | `smoke` 或 `full` | 默认 `smoke`；不继承 | 区分快速流程检查与显式选择的完整验收。 |
| `OUTPUT_ROOT` | 相对仓库根目录的路径 | `notebook_outputs/cone-detection/` | 根据所选练习确定；不继承 | 将生成文件保存在已忽略的 notebook 输出目录中。 |
| `MODE.rollout_steps` | 步 | smoke：`24`；full：`1000` | 由 `RUN_MODE` 选择；不继承 | 限制后续无噪声和有噪声仿真的步数。 |
| `MODE.animation_frame_step` | 状态转移/帧 | smoke：`4`；full：`6` | 由 `RUN_MODE` 选择；不继承 | smoke 每隔四次、full 每隔六次状态转移取一帧动画。 |

课程的 [notebook 设置指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/setup_notebooks.md)、[道路指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/road.md)、[观测指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/observations.md)、[评估工具指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/evaluation_utilities.md) 和 [可视化指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/visualization.md) 介绍本参考示例所用的公开接口及其边界。


In [3]:
RUN_MODE = "full"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
OUTPUT_ROOT = Path("notebook_outputs") / "cone-detection"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODE = {
    "smoke": {"rollout_steps": 24, "animation_frame_step": 4},
    "full": {"rollout_steps": 1000, "animation_frame_step": 6},
}[RUN_MODE]


In [4]:
import dreamgym
dreamgym_source = Path(dreamgym.__file__).resolve()
if IS_COLAB and not dreamgym_source.is_relative_to(COLAB_CHECKOUT.resolve()):
    raise RuntimeError("Dream Gym resolved outside the selected Colab checkout")

from dataclasses import dataclass
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from dreamgym.envs import Road
from dreamgym.evaluation import MetricSet
from dreamgym.evaluation.autonomous_driving import (
    compute_autonomous_driving_metrics,
    metric_by_name,
    simulate_policy,
)
from dreamgym.visualization.autonomous_driving import (
    animate_simulation_trajectory,
    plot_road_from_spec,
    plot_simulation_results,
    save_figure,
    save_road_plot,
    save_trajectory_animation,
)


ModuleNotFoundError: No module named 'dreamgym'

## 1) 定义路锥道路


**定义路锥布局**

我们使用之前的小车道路：10 m 直道、半径 5 m 且向左转 45° 的弯道，以及一段 8 m 直道。车道宽 1 m。道路的默认限速为 1.8 km/h（0.5 m/s），与后面固定驱动指令示例的大致行驶速度相符。

`cone_paths` 定义两侧边界及其布置配置。在本例中，蓝色代表左侧，黄色代表右侧；这是本例的约定，并非通用的颜色规则。对于这条居中的车道，左侧边界的 `lane_line_index` 为 +1，右侧为 −1。

`regular` 配置每隔 0.5 m 放置一个路锥。右侧边界还有 `sparse_displaced` 配置：间距为 0.8–1.2 m，横向偏移为 −0.15–0.15 m。这些是路锥的实际位置变化，并非传感器噪声。距离单位为米。


In [ ]:
CONE_LANE_WIDTH_M = 1.0
CONE_ROAD_PLOT_SEED = 0

CONE_REGULAR_PROFILE = {
    "inter_cone_spacing": {"distribution": "fixed", "value_m": 0.5},
}
CONE_IRREGULAR_PROFILE = {
    "inter_cone_spacing": {
        "distribution": "uniform",
        "lower_bound_m": 0.8,
        "upper_bound_m": 1.2,
    },
    "lateral_displacement": {
        "distribution": "uniform",
        "lower_bound_m": -0.15,
        "upper_bound_m": 0.15,
    },
}
CONE_PATHS = {
    "left_boundary": {
        "placement": {"mode": "lane_line", "lane_line_index": 1},
        "color": "blue",
        "radius_m": 0.08,
        "profiles": {"regular": CONE_REGULAR_PROFILE},
    },
    "right_boundary": {
        "placement": {"mode": "lane_line", "lane_line_index": -1},
        "color": "yellow",
        "radius_m": 0.08,
        "profiles": {
            "regular": CONE_REGULAR_PROFILE,
            "sparse_displaced": CONE_IRREGULAR_PROFILE,
        },
    },
}


**构建道路**

每段道路通过 `active_cone_paths` 选择路锥配置：

1. 第一段直道：两侧都是规则排列的路锥。
2. 左弯道：只有左侧有路锥。
3. 最后一段直道：左侧规则排列，右侧稀疏排列且有横向偏移。

显式设置映射会替换上一段的选择；省略时则继承上一段的选择。这里明确写出三段的映射。车道宽度从第一段沿用至后续路段。


In [ ]:
CONE_ROAD_SPEC = {
    "defaults": {"speed_limit_kph": 1.8},
    "cone_paths": CONE_PATHS,
    "elements": [
        {
            "geometry": {"type": "straight", "length_m": 10.0},
            "cross_section_width_interpolation": "constant",
            "lanes": {
                "reference_line_role": "lane_center",
                "reference_lane": {
                    "width_m": {"start": CONE_LANE_WIDTH_M, "end": CONE_LANE_WIDTH_M},
                    "surface": "tarmac",
                },
            },
            "active_cone_paths": {
                "left_boundary": "regular",
                "right_boundary": "regular",
            },
        },
        {
            "geometry": {
                "type": "circular_arc",
                "curvature_per_m": 1.0 / 5.0,
                "sweep_angle_deg": 45.0,
            },
            "active_cone_paths": {"left_boundary": "regular"},
        },
        {
            "geometry": {"type": "straight", "length_m": 8.0},
            "active_cone_paths": {
                "left_boundary": "regular",
                "right_boundary": "sparse_displaced",
            },
        },
    ],
}
cone_road = Road(road_spec=CONE_ROAD_SPEC)
resolved_cones = cone_road.resolve_cones(cone_seed=CONE_ROAD_PLOT_SEED)


**绘图并检查**

随机种子使不规则布局可以重现。此预览使用一套确定后的路锥布局；稍后运行策略时，我们还会控制环境的路锥种子。

**观察要点：** 找出三个路段。弯道上消失的是哪一侧？在最后一段直道上，区分沿边界方向的较大间距与横跨边界方向的偏移。

更多选项见[道路路锥配置指南](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/road.md#cones)。


In [ ]:
cone_road_figure, cone_road_axis = plt.subplots(figsize=(10, 5))
cone_road.plot_road_on_matplotlib_axis(
    cone_road_axis,
    resolved_cones=resolved_cones,
)
cone_road_axis.set_title("Small-car road: regular cones → one side → irregular right side")
cone_road_axis.set_xlabel("world x [m]")
cone_road_axis.set_ylabel("world y [m]")
cone_road_axis.set_aspect("equal", adjustable="box")
plt.show()
plt.close(cone_road_figure)


## 2) 定义 1:10 比例小车的车辆参数


**仿真中的系统项目小车**

本 notebook 使用系统项目 1:10 比例小车的简化自行车模型，开发并测试沿路锥行驶策略。

仿真中轴距为 0.33 m，质量为 3 kg。质心到前轴和后轴的距离分别为轴距的 60% 和 40%。我们用矩形车身近似偏航转动惯量，并指定绘制小车所需的车身尺寸。

**转向与驱动响应**

在此模型中，转向请求限制在 ±45°，最大转向速度为每秒 90°，转向偏置为零。转换阈值使模型在本 notebook 使用的速度范围内保持运动学模式。

驱动指令归一化到 −1 至 1。配置的电机力增益为 10 N，二次阻力系数为 1 kg/m。这个初始模型使用线性驱动响应，没有指令死区。零驱动会去除电机力，但不会让行驶中的小车瞬间停下。

这些设置仅作为仿真模型的起点，不应被视为实体小车的完整标定。

**定义车辆配置**

下面的字典定义 `bicycle_model_config`，第 4 节会将其传给环境。距离单位为米，角度从度转换为弧度。


In [ ]:
bicycle_model_config = {
    "axle_geometry": {
        "front_axle_distance_from_center_of_mass_m": 0.60 * 0.33,
        "rear_axle_distance_from_center_of_mass_m": 0.40 * 0.33,
    },
    "mass_properties": {
        "mass_kg": 3.0,
        "yaw_moment_of_inertia_kg_m2": (1.0 / 12.0) * 3.0 * (0.40**2 + 0.25**2),
    },
    "longitudinal_force_model": {
        "normalized_motor_command_force_gain_n": 10.0,
        "quadratic_aerodynamic_drag_coefficient_kg_per_m": 1.0,
        "direction_change_latch": {
            "forward_to_reverse": True,
        },
    },
    "steering": {
        "physical_front_wheel_angle_offset_from_request_rad": 0.0,
        "requested_front_wheel_angle_magnitude_limit_rad": np.deg2rad(45.0),
        "physical_front_wheel_angle_rate_bounds_rad_per_s": {
            "lower": np.deg2rad(-90.0),
            "upper": np.deg2rad(90.0),
        },
    },
    "kinematic_dynamic_transition": {
        "body_longitudinal_speed_range_mps": {
            "lower": 500.0 / 3.6,
            "upper": 600.0 / 3.6,
        },
    },
    "body_geometry": {
        "front_extent_from_center_of_mass_m": (0.60 * 0.33) * 1.5,
        "rear_extent_from_center_of_mass_m": (0.40 * 0.33) * 1.5,
        "width_m": 0.25,
    },
}


**制动与倒车锁存机制**

从前进到倒车的方向锁存机制使仿真驱动指令更接近系统项目小车的行为。小车前进时，负驱动指令会让它制动并保持静止；持续保持同一指令不会让它开始倒车。之后如需倒车，应在小车静止时先请求空挡以解除锁存，再发出新的负驱动指令。

此处只启用前进到倒车的锁存机制，倒车到前进的锁存机制保持关闭。


## 3) 定义路锥检测配置


**小车能检测到什么？**

道路配置决定路锥实际位于何处。观测配置决定小车能检测到哪些路锥，以及如何向策略报告其位置。

起始设置为水平视场角 80°、向前检测上限 4 m。检测器最多返回 24 个路锥。这些只是仿真的初始设置，不是实体相机经标定的规格。

策略只接收路锥检测结果：相对于车身坐标系的前向和左向位置（单位：米）、颜色 ID，以及有效检测数量。坐标和颜色数组的容量固定；只有前 `count` 个条目有效，其余是填充数据，不代表额外路锥。

我们使用默认的固定相机安装方式：相机位于车身坐标系原点，朝向正前方。采集噪声在相机坐标系中配置，请求输出的坐标则在车身坐标系中。在这种安装方式下，两个坐标系重合。

**先不添加位置噪声**

`CONE_DETECTION_NOISE_STD_M` 设置前向和左向位置噪声各自的标准差。先设为零，便于理解第一次仿真。启用噪声后，固定的噪声种子可用于重复比较。


In [ ]:
CONE_DETECTION_NOISE_STD_M = 0.0
CONE_NOISE_SEED = 20260811

CONE_OBSERVATION_CONFIG = {
    "cone_detections": {
        "maximum_detection_count": 24,
        "acquisition": {
            "horizontal_field_of_view_deg": 80.0,
            "forward_position_upper_bound_m": 4.0,
            "position_error_in_camera_frame": {
                "forward": {
                    "noise_standard_deviation_m": CONE_DETECTION_NOISE_STD_M,
                },
                "left": {
                    "noise_standard_deviation_m": CONE_DETECTION_NOISE_STD_M,
                },
            },
        },
        "positions_in_body_frame_m": {
            "destination": "observation",
            "scaling_factor": 1.0,
        },
        "color_ids": {"destination": "observation"},
        "count": {"destination": "observation"},
    },
    "fixed_noise_seed": CONE_NOISE_SEED,
}


**稍后尝试：添加位置噪声**

运行第 6 节的基准沿路锥行驶仿真后，返回这里，为 `CONE_DETECTION_NOISE_STD_M` 设置一个较小的非负值，例如 `0.03`（3 cm）。先预测路锥位置的不确定性会如何影响转向，再与无噪声结果比较轨迹。

重新运行此配置单元、在第 4 节重新创建环境，并重新运行第 5–6 节的策略和仿真单元。只修改这个变量不会更新已创建的环境。比较时保持道路、策略设置和随机种子不变。你可以继续调整噪声；示例数值不是经过测量的传感器规格。

位置噪声改变报告的路锥坐标，而第 1 节的不规则路锥配置改变的是其实际位置。请区分这两种变化来源。


## 4) 创建环境


**组合模型、道路和检测器**

现在把第 1 节的道路、第 2 节的车辆和第 3 节的观测组合起来。

小车从道路起点出发，沿道路方向行驶，初速度为 0.5 m/s；初始横向偏移在 −0.1 至 0.1 m 之间抽样。初始横向速度、偏航角速度和转向角采用默认零值。固定的重置种子使起点可重复。

仿真每步前进 0.05 s，使用 RK4 和一个积分子步。我们设置较宽松的终止范围：平面速度为 0–10 m/s，相对于目标线的绝对横向误差不超过 20 m。这些是仿真停止条件，并非期望的运行界限或车道保持成功标准。仿真时长会在第 6 节选择。

此处关闭渲染；第 6 节会对记录的仿真结果绘图并生成动画。更改前面的配置后，请重新运行此单元来构建新环境。


In [ ]:
CONE_RESET_SEED = 0

initial_state_config = {
    "pose": {
        "mode": "road_relative",
        "anchor": "reference_line",
        "progress": {"unit": "metres", "lower": 0.0, "upper": 0.0},
        "lateral_offset_m": {"lower": -0.1, "upper": 0.1},
        "heading_error_rad": {"lower": 0.0, "upper": 0.0},
    },
    "body_motion": {
        "longitudinal_velocity": {
            "source": "absolute",
            "value_mps": {"lower": 0.5, "upper": 0.5},
        },
    },
}
integration_config = {
    "method": "rk4",
    "environment_step_duration_s": 0.05,
    "substep_count": 1,
}
termination_config = {
    "planar_speed_bounds_mps": {"lower": 0.0, "upper": 10.0},
    "absolute_lateral_error_from_target_line_upper_bound_m": 20.0,
}

# Close the previous instance when rerunning this cell.
if "env" in globals():
    env.close()

env = gym.make(
    "dreamgym/autonomous_driving_env",
    road_spec=CONE_ROAD_SPEC,
    bicycle_model_config=bicycle_model_config,
    observation_config=CONE_OBSERVATION_CONFIG,
    initial_state_config=initial_state_config,
    integration_config=integration_config,
    termination_config=termination_config,
    observation_format="dict",
    render_mode=None,
)


**检查一次观测**

重置环境会开始新的回合，并返回观测和诊断信息。下面只显示有效的路锥条目；每个固定容量数组中剩余的部分都是填充数据。

**观察要点：** 前向和左向位置相对于小车测量，不是世界坐标。每个颜色 ID 对应同一数组索引处的路锥。策略观测只包含路锥检测结果，不包含小车在道路上的真实位置。后续章节会继续使用 `env`。


In [ ]:
observation, info = env.reset(seed=CONE_RESET_SEED)
cone_count = int(observation["cone_detections_count"][0])
print(f"Valid cone detections: {cone_count}")
display({
    "forward position [m]": observation[
        "cone_detections_forward_positions_in_body_frame_m"
    ][:cone_count],
    "left position [m]": observation[
        "cone_detections_left_positions_in_body_frame_m"
    ][:cone_count],
    "colour ID": observation["cone_detections_color_ids"][:cone_count],
})


## 5) 沿路锥行驶策略


**从检测到的路锥生成动作**

沿路锥标记路线行驶的系统项目小车需要把检测结果转换为转向和驱动指令。这里使用检测到的路锥位置和颜色来估计路线走向，并朝车道中心转向。策略只使用路锥观测，不通过 `info` 读取小车在道路上的真实位置。

策略分为五步：

1. 读取有效检测并按颜色分类：在此道路上，蓝色标记左边界，黄色标记右边界。
2. 如果一侧边界至少有两个前向位置不同的路锥，就为它拟合一条直线，并计算该直线在前瞻距离处的位置。
3. 对两侧边界的估计值取平均，得到中心位置。若只看到一侧边界，就将其估计值平移半个车道宽度。这样即使弯道上只有左侧路锥，小车也能继续沿路线行驶。
4. 用估计中心的横向偏移乘以转向增益，限制转向角，再转换为归一化转向指令。
5. 施加固定的驱动指令。此策略不使用测得的速度反馈。

横向偏移为正表示左侧，因此正转向请求会使小车左转。直线拟合只是对附近边界的近似；急弯、有噪声的检测或分布不佳的路锥都会降低其准确度。如果单侧边界相对小车倾斜，平移半个车道宽度同样只是近似。

如果两侧边界都无法估计，策略会返回零驱动和零转向。零驱动会去除电机力，但不会让小车瞬间停下。请思考当路锥检测不可靠或消失时，你的系统项目策略应该如何响应，以及如何测试这种响应。


In [ ]:
class ConeFollowingPolicy:
    def __init__(self, lane_width_m, lookahead_m, steering_gain_rad_per_m,
                 drive_command, steering_limit_rad):
        # Store the settings chosen in the parameter cell below.
        self.lane_width_m = lane_width_m
        self.lookahead_m = lookahead_m
        self.steering_gain_rad_per_m = steering_gain_rad_per_m
        self.drive_command = drive_command
        self.steering_limit_rad = steering_limit_rad

    def boundary_at_lookahead(self, forward_m, left_m, colour_ids, colour_id):
        # Select this boundary colour, keeping finite positions ahead of the car.
        usable = (
            (colour_ids == colour_id) & (forward_m >= 0.0)
            & np.isfinite(forward_m) & np.isfinite(left_m)
        )
        x, y = forward_m[usable], left_m[usable]
        # A line fit needs at least two different forward positions.
        # None means we cannot estimate this boundary; it is different from zero.
        if len(x) < 2 or np.unique(x).size < 2:
            return None
        # Fit left_position = slope * forward_position + intercept.
        slope, intercept = np.polyfit(x, y, 1)
        # Predict the boundary's lateral position at our chosen distance ahead.
        return slope * self.lookahead_m + intercept

    def compute_action(self, observation, info, terminated, truncated):
        # Match the simulator interface; info is deliberately unused.
        # If the episode has ended, return neutral drive and steering.
        if terminated or truncated:
            return np.zeros(2, dtype=np.float32)

        # Ignore padding: use only the first count entries.
        count = int(observation["cone_detections_count"][0])
        forward_m = observation["cone_detections_forward_positions_in_body_frame_m"][:count]
        left_m = observation["cone_detections_left_positions_in_body_frame_m"][:count]
        colour_ids = observation["cone_detections_color_ids"][:count]

        # Estimate both boundaries at the same lookahead distance.
        # These colour-to-side assignments match the road in section 1.
        left_boundary = self.boundary_at_lookahead(
            forward_m, left_m, colour_ids, Road.CONE_COLOR_TO_ID["blue"]
        )
        right_boundary = self.boundary_at_lookahead(
            forward_m, left_m, colour_ids, Road.CONE_COLOR_TO_ID["yellow"]
        )

        # Both sides visible: aim halfway between them.
        if left_boundary is not None and right_boundary is not None:
            centre_left_m = (left_boundary + right_boundary) / 2.0
        elif left_boundary is not None:
            # The centre is to the right of the left boundary.
            centre_left_m = left_boundary - self.lane_width_m / 2.0
        elif right_boundary is not None:
            # The centre is to the left of the right boundary.
            centre_left_m = right_boundary + self.lane_width_m / 2.0
        else:
            # No usable boundary: remove drive force and request straight steering.
            # This does not instantly stop the car.
            return np.zeros(2, dtype=np.float32)

        # Positive centre offset requests a left turn; negative requests right.
        # Gain converts metres of offset to radians; clip enforces the angle limit.
        steering_rad = np.clip(
            self.steering_gain_rad_per_m * centre_left_m,
            -self.steering_limit_rad, self.steering_limit_rad,
        )
        # The environment expects normalized steering in [-1, 1], not radians.
        steering_command = steering_rad / self.steering_limit_rad
        # Action order is [drive, steering]. Drive is fixed, not speed feedback.
        # float32 matches the environment's action-space dtype.
        return np.array([
            np.clip(self.drive_command, -1.0, 1.0), steering_command,
        ], dtype=np.float32)


**选择初始值**

前瞻距离从 1 m 开始，处于检测器 4 m 的前向上限内；转向增益设为估计偏移每米 0.5 rad。按当前车辆配置，0.025 的驱动指令会产生 0.25 N 电机力；在设置的阻力下，直线行驶时这大致能平衡 0.5 m/s 速度对应的阻力（稳态速度约为 √(10 × 驱动指令) m/s）。它是驱动设置，而非速度指令。

这些是可编辑的初始值，并非优化后的控制器。完成第 6 节仿真后，每次改变一个值并比较路径。更长的前瞻距离或更大的增益会改变小车对估计边界的响应强度。修改数值后，重新运行下面的单元，让策略实例使用新值。


In [ ]:
CONE_LOOKAHEAD_DISTANCE_M = 1.0
CONE_STEERING_GAIN_RAD_PER_M = 0.5
CONE_DRIVE_COMMAND = 0.025

cone_following_policy = ConeFollowingPolicy(
    lane_width_m=CONE_LANE_WIDTH_M,
    lookahead_m=CONE_LOOKAHEAD_DISTANCE_M,
    steering_gain_rad_per_m=CONE_STEERING_GAIN_RAD_PER_M,
    drive_command=CONE_DRIVE_COMMAND,
    steering_limit_rad=bicycle_model_config["steering"][
        "requested_front_wheel_angle_magnitude_limit_rad"
    ],
)


## 6) 仿真沿路锥行驶策略


**运行仿真**

`simulate_policy` 使用选定的种子重置 `env`，然后反复向策略请求动作并推进仿真。它记录结果以供绘图；如果环境结束回合，就会提前停止。

smoke 模式运行 24 步（最长 1.2 s），用于检查流程能否运行。若要查看整条路线，在“准备与导入”中选择 `RUN_MODE = "full"`，再重新运行运行设置单元。full 模式最多运行 1000 步（50 s）；如果跟踪成功，以约 0.5 m/s 的速度行驶（约需 44 s），足以覆盖这条约 22 m 的道路。


In [ ]:
cone_results = simulate_policy(
    env,
    MODE["rollout_steps"],
    cone_following_policy,
    seed=CONE_RESET_SEED,
)


**检查轨迹和时间序列**

观察小车经过规则路锥路段、单侧路锥弯道和最后的不规则路锥直道。比较轨迹与随时间变化的速度、转向和横向误差。仿真完成本身不代表跟踪良好：宽松的终止范围允许小车驶离车道。

速度图以 km/h 为单位，并显示道路建议速度 1.8 km/h 的参考线。这个值等于 0.5 m/s，选择它是为了匹配固定驱动指令示例的大致工作速度。它仍只是参考值，不是速度指令；此策略没有使用测得的速度反馈。


In [ ]:
cone_result_figures = plot_simulation_results(env, cone_results, should_plot_reward=False)
for figure in cone_result_figures.values():
    display(figure)
    plt.close(figure)


**观看小车沿路锥行驶**

使用动画控件重放运动过程，并将其与图表联系起来。动画每隔四个仿真步显示一帧；这只改变回放采样，不改变控制器的更新频率。


In [ ]:
from IPython.display import HTML

cone_animation = animate_simulation_trajectory(
    env,
    cone_results,
    frame_step=MODE["animation_frame_step"],
    figure_title="Following the cone-marked route",
    zoom_width_m=6.0,
    zoom_height_m=6.0,
)
display(HTML(cone_animation.to_jshtml()))
plt.close(cone_animation._fig)


**尝试修改策略**

先预测改变前瞻距离或转向增益的效果。在第 5 节修改一个参数，重新创建 `cone_following_policy`，然后重新运行本节。保持随机种子不变，比较弯道和最后直道上的跟踪效果。

若要研究位置噪声，按第 3 节的说明重新运行相关单元。除了路径，还要观察转向的变化。找出策略表现不佳的位置，并说明你会如何修改系统项目控制器。


## 7) 在道路上添加障碍物


**路线上的物体**

沿路锥行驶只是路线导航的一部分：系统项目小车还可能在路上遇到物体。这里沿路线加入两个静止的矩形障碍物，道路和路锥保持不变。

`geometry` 定义每个矩形的长度和宽度，单位为米。`placement` 用沿道路参考线的距离和横向偏移（向左为正）定位其中心。`heading_offset_rad` 指定相对于道路方向的旋转角度；零表示与道路同向。

最后一段直道从沿道路约 13.93 m 处开始。在 12 m 处，右侧路锥缺失的弯道上，有一个宽 0.3 m 的障碍物位于参考线右侧 0.85 m 处。其最近边缘约在车道外侧 0.2 m，因此小车有空间沿左侧路锥行驶。在 20 m 处，一个居中的 0.45 m 宽障碍物位于路锥边界内，挡住小车预定路径。这样就形成先经过路边物体、再接近需要作出响应的障碍物的过程。我们先复制道路配置，使基准道路仍可用于比较。


In [ ]:
from copy import deepcopy

ROAD_SPEC_WITH_OBSTACLES = deepcopy(CONE_ROAD_SPEC)
ROAD_SPEC_WITH_OBSTACLES["obstacles"] = [
    {
        "geometry": {"type": "rectangle", "length_m": 0.4, "width_m": 0.3},
        "placement": {
            "reference_line_progress_m": 12.0,
            "lateral_offset_m": -0.85,
            "heading_offset_rad": 0.0,
        },
    },
    {
        "geometry": {"type": "rectangle", "length_m": 0.4, "width_m": 0.45},
        "placement": {
            "reference_line_progress_m": 20.0,
            "lateral_offset_m": 0.0,
            "heading_offset_rad": 0.0,
        },
    },
]
road_with_obstacles = Road(road_spec=ROAD_SPEC_WITH_OBSTACLES)
resolved_cones_with_obstacles = road_with_obstacles.resolve_cones(
    cone_seed=CONE_ROAD_PLOT_SEED,
)


**检查障碍物位置**

全路段视图保留整体背景，近景图则更便于看清小障碍物。重复使用路锥随机种子，可使路锥实际布局与第 1 节保持一致。

小车沿车道行驶时可以经过哪个物体？哪个物体挡住了去路？检测到附近有东西，是否总意味着应当停车？仅根据最小 LiDAR 距离制定规则，也可能对路边物体作出响应；添加感知与停车逻辑时，我们会再次研究这个区别。现在障碍物已写入道路配置，但传感器和策略尚未改变。后续章节会引入 LiDAR 和单独的环境。


In [ ]:
obstacle_figure, obstacle_axes = plt.subplots(1, 2, figsize=(12, 5))
for axis in obstacle_axes:
    road_with_obstacles.plot_road_on_matplotlib_axis(
        axis,
        resolved_cones=resolved_cones_with_obstacles,
        should_render_obstacles=True,
    )
    axis.set_xlabel("world x [m]")
    axis.set_ylabel("world y [m]")
    axis.set_aspect("equal", adjustable="box")

obstacle_axes[0].set_title("Cone-marked road with obstacles")
obstacle_axes[1].set_title("Bend-side object and final obstacle")
obstacle_axes[1].set_xlim(10.5, 19.5)
obstacle_axes[1].set_ylim(-1.5, 7.5)
plt.tight_layout()
plt.show()
plt.close(obstacle_figure)


## 8) 添加基于 LiDAR 的障碍物观测


**测量到障碍物的距离**

小车仍使用路锥检测结果来跟随路线。现在加入第 7 节障碍物的 LiDAR 测量，使策略也能利用附近物体的信息。路锥观测保持不变。

我们保留路锥观测，并添加 `obstacle_lidar_ranges_m`。每个条目表示从传感器到该光束遇到的第一个障碍物表面的距离，单位为米。没有回波的光束报告最大量程，而不是零；读数等于最大量程时，无法区分障碍物恰好位于量程边界和没有回波。

本例使用朝前、覆盖 180° 视场的 61 条光束，最大量程 4 m，最小量程 0.05 m。光束间隔 3°：数组索引 0 指向右侧（−90°），索引 30 指向正前方（0°），索引 60 指向左侧（+90°）。传感器位于车身坐标系原点，并与小车对齐。

我们先使用无噪声的距离测量。下面的设置定义本次仿真所用传感器；小于 0.05 m 的测量值会报告为 0.05 m。


In [ ]:
OBSERVATION_CONFIG_WITH_LIDAR = deepcopy(CONE_OBSERVATION_CONFIG)
OBSERVATION_CONFIG_WITH_LIDAR["obstacle_lidar"] = {
    "model": "rplidar_c1",
    "range_encoding": "metres",
    "mounting_pose_in_body_frame": {
        "forward_from_center_of_mass_m": 0.0,
        "left_from_center_of_mass_m": 0.0,
        "heading_offset_from_body_rad": 0.0,
    },
    "acquisition": {
        "beam_count": 61,
        "horizontal_field_of_view_deg": 180.0,
        "minimum_range_m": 0.05,
        "maximum_range_m": 4.0,
        "noise_standard_deviation_m": 0.0,
        "bias_m": 0.0,
        "resolution_m": 0.0,
    },
}


**预测扫描结果**

小车接近弯道上的路边物体时，哪些光束可能检测到它？与小车接近路径中央的障碍物时相比，这些回波读数有何不同？

最小距离只能说明附近有物体，不能说明它一定挡住路线。因此，仅根据最小距离停车的规则也可能对路边物体作出响应。判断物体是否位于行驶路径上时，光束方向和车身宽度都很重要。

本单元只定义额外观测。第 9 节会创建新环境；第 10 节会运行现有的沿路锥行驶策略，该策略起初会忽略 LiDAR。第 11 节会引入停车条件。修改这些传感器设置后，请重新创建障碍物环境。


## 9) 创建含障碍物的新环境


**创建障碍物环境**

我们沿用第 4 节的车辆、初始条件和仿真设置。只有道路与观测配置发生变化：此环境包含障碍物，除了路锥检测结果，还提供 LiDAR 距离。

新环境命名为 `env_with_obstacles`，使原来的 `env` 仍可用于比较。更改障碍物或 LiDAR 配置后，请重新运行此单元。


In [ ]:
# Close the previous obstacle environment when rerunning this cell.
if "env_with_obstacles" in globals():
    env_with_obstacles.close()

env_with_obstacles = gym.make(
    "dreamgym/autonomous_driving_env",
    road_spec=ROAD_SPEC_WITH_OBSTACLES,
    bicycle_model_config=bicycle_model_config,
    observation_config=OBSERVATION_CONFIG_WITH_LIDAR,
    initial_state_config=initial_state_config,
    integration_config=integration_config,
    termination_config=termination_config,
    observation_format="dict",
    render_mode=None,
)


**检查起点处的观测**

使用与基准仿真相同的种子重置环境。路锥检测结果仍可用于转向。两个障碍物起初都在 LiDAR 的 4 m 量程之外，因此正前方读数和最小距离都应为 4 m。小车接近障碍物时，读数会变化。

对于本例朝前的 61 光束扫描，中间条目指向正前方。最小值会考虑所有光束方向，因此也可能反映小车旁边的物体。


In [ ]:
obstacle_observation, obstacle_info = env_with_obstacles.reset(seed=CONE_RESET_SEED)
lidar_ranges_m = obstacle_observation["obstacle_lidar_ranges_m"]
forward_beam_index = len(lidar_ranges_m) // 2

display({
    "valid cone detections": int(obstacle_observation["cone_detections_count"][0]),
    "forward LiDAR range [m]": float(lidar_ranges_m[forward_beam_index]),
    "minimum LiDAR range [m]": float(np.min(lidar_ranges_m)),
})


环境现在提供障碍物测量值，但 `cone_following_policy` 仍只使用路锥。第 10 节会在此环境中运行这一策略；第 11 节会引入停车响应。


## 10) 在障碍物环境中仿真


**在有障碍物时运行同一策略**

小车现在会收到 LiDAR 测量值，但策略仍只使用路锥检测结果。运行仿真，观察为什么增加传感器不会自动改变小车的行为。

使用与第 6 节相同的策略、重置种子和仿真时长。在“准备与导入”中选择 full 模式并重新运行运行设置单元，才能到达障碍物；smoke 模式只检查起始路段。辅助函数会在运行前重置 `env_with_obstacles`。

此次运行还会记录每个状态下实际测得的 LiDAR 扫描，以便在动画中重放。记录保留每次扫描测得的距离和传感器位姿；之后不会重新采样传感器，也不会根据障碍物几何形状重建测量值。


In [ ]:
obstacle_results = simulate_policy(
    env_with_obstacles,
    MODE["rollout_steps"],
    cone_following_policy,
    seed=CONE_RESET_SEED,
    should_save_obstacle_lidar=True,
)


**检查轨迹和时间序列**

将这些结果与第 6 节比较。策略没有改变，因此额外的 LiDAR 观测此时不会影响其动作。

在此仿真器中，障碍物接触会被记录，但不会让小车在物理上停下或改变方向。因此轨迹可能穿过障碍物。这不代表成功避障：控制器必须利用可用测量值作出响应。速度图沿用第 6 节说明的道路建议速度参考线。


In [ ]:
obstacle_result_figures = plot_simulation_results(
    env_with_obstacles, obstacle_results, should_plot_reward=False,
)
for figure in obstacle_result_figures.values():
    display(figure)
    plt.close(figure)


**观看遇到障碍物的过程**

观察小车经过弯道上的路边物体，并接近最后的障碍物。使用回放控件检查其转向或驱动是否对任一物体作出响应。

蓝色 LiDAR 叠加图重放完整的已记录扫描，尽管临时停车策略只使用狭窄的前向光束子集。因此，看到路边回波不代表策略曾用它来请求制动。


In [ ]:
from IPython.display import HTML

obstacle_animation = animate_simulation_trajectory(
    env_with_obstacles,
    obstacle_results,
    frame_step=MODE["animation_frame_step"],
    figure_title="Cone following with obstacles",
    zoom_width_m=6.0,
    zoom_height_m=6.0,
    should_render_obstacle_lidar=True,
)
display(HTML(obstacle_animation.to_jshtml()))
plt.close(obstacle_animation._fig)


**需要改变什么？**

- 小车能否在不驶离车道的情况下经过路边物体？
- 它会对最后的障碍物作出响应，还是继续沿路锥路线前进？
- 你会在 `compute_action` 的什么位置使用 LiDAR 距离来改变驱动指令？

为了完成这次比较，请保持策略不变。第 11 节会引入可供添加和测试的停车条件。


## 11) 练习：为策略添加物体检测与停车功能


**练习：扩展并测试策略**

在本练习中，你将修改第 5 节的策略，使其使用 LiDAR 距离，并在第 10 节评估响应。下面的代码片段是实现的起点。

**根据障碍物距离改变驱动指令**

小车已经能接收 LiDAR 距离。请在 `compute_action` 中添加基于距离的响应，使附近物体能够改变驱动指令。

在第 5 节，用下面的代码片段替换最后的 `return np.array(...)` 代码块。将它保留在 `compute_action` 内，并放在计算 `steering_command` 之后。局部变量 `drive_command` 在每次调用时都从你选定的设置开始，不会覆盖 `self.drive_command`。

```python
# 从通常的驱动指令开始，然后检查附近障碍物。
stop_threshold_m = 1.0  # 用于探索的初始值，并非有保证的停车距离。
drive_command = self.drive_command
lidar_ranges_m = observation["obstacle_lidar_ranges_m"]
if np.min(lidar_ranges_m) < stop_threshold_m:
    drive_command = -1.0  # 通过方向锁存机制请求制动。

# 制动时继续保持沿路锥行驶的转向指令。
return np.array([
    np.clip(drive_command, -1.0, 1.0), steering_command,
], dtype=np.float32)
```

重新运行第 5 节的策略类单元及参数/实例化单元，再以 full 模式重新运行第 10 节。修改后的策略需要 `env_with_obstacles` 提供的 LiDAR 观测；第 6 节原来的仅有路锥的 `env` 不提供这些观测。再次使用原环境前，请恢复原来的返回代码块。

**制动请求能让小车及时停下吗？**

启用第 2 节的前进到倒车方向锁存机制后，小车前进时的负驱动指令会作为制动请求。小车停下后，继续保持负指令会让它停住，而不会开始倒车。静止时选择空挡可解除锁存；之后再次发出负指令便可选择倒车。

这个距离规则仍不是完整的停车策略。一旦最小距离高于阈值，通常的正驱动指令就会恢复，因此策略不会记住先前的停车决定。1 m 阈值只是有待研究的初始值，不保证停车距离。如第 10 节所述，在此仿真器中，接触障碍物也不会从物理上阻止运动。

- 预测增大或减小阈值会如何改变开始制动的时机，然后每次只测试一种改动。
- 即使车道仍畅通，路边物体是否会引起响应？最小距离本身无法判断物体是否位于小车行驶路径上。

> **提示：** `np.min(lidar_ranges_m)` 会考虑整个 180° 扫描范围内的每条光束，包括小车旁边的物体。如果路边物体导致不必要的制动，可考虑只使用数组中央附近、方向大致朝前的光束。先从狭窄扇区开始，再研究扩大或缩小扇区如何改变对最后一个障碍物的检测。固定的前向扇区仍只是一种启发式方法，转弯时尤其如此。

> 第 8 节介绍 LiDAR 光束顺序，第 9 节计算朝前光束的索引。可考虑在 `lidar_ranges_m` 中截取该索引附近的一小段，而不是对整次扫描求最小值。

- 除动画外，还要检查速度、驱动指令和障碍物接触记录。速度是否在接触前降为零？如果物体离开扫描范围，正驱动指令是否恢复？
- 对于系统项目策略，请思考如何通过选择朝前光束、考虑停车距离以及记住停车决定来改进响应。

单纯增大阈值不能保证停车。请设计并测试所需的响应，包括判断哪些光束方向与预期路径有关。上面的简短示例旨在帮助你开始这一设计过程。


## 12) 性能指标与不确定性扫描


在 Upskilling 中介绍的性能指标和不确定性扫描流程，也可应用于本系统项目仿真。借助它们，你可以超越单次轨迹或动画来评估策略。

请选择能反映团队预期行为的指标，例如跟踪精度、行驶进度和障碍物接触情况。然后通过不确定性扫描，研究起点位置或传感器噪声等条件改变时性能如何变化。比较不同策略时，请保持比较条件一致。

实现这些流程时，可参考之前的 Upskilling notebooks。请说明结果反映了策略的哪些优势、局限和可靠性特征，并用这些证据指导下一步改进。
